# Predict Customer Churn
## Score: .91279

In [1]:
USE_ORIGINAL_DATA = False
USE_TARGET_ENCODING = False
USE_RF_ET = False
N_FOLDS = 10
N_SEEDS = 2
USE_MLP = True
USE_CALIBRATION = True
USE_PSEUDOLABELING = True
PSEUDO_WEIGHT = 0.15

In [2]:
import numpy as np
import pandas as pd

train = pd.read_csv("playground-series-s6e3/train.csv")
test = pd.read_csv("playground-series-s6e3/test.csv")
test_ids = test['id']

if USE_ORIGINAL_DATA:
    original = pd.read_csv("original_data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
    original = original.drop(columns=['customerID'])
    original = original[train.columns.drop('id')]
    train_combined = pd.concat([train.drop(columns=['id']), original], ignore_index=True)
    sample_weights = np.array([1.0] * len(train) + [0.5] * len(original))
else:
    train_combined = train.drop(columns=['id'])
    sample_weights = None

y_full = train_combined['Churn'].map({'Yes': 1, 'No': 0})
X_full = train_combined.drop(columns=['Churn'])
X_test_raw = test.drop(columns=['id'])
print(f"Train: {len(X_full)}, Sample weights: {sample_weights is not None}")

Train: 594194, Sample weights: False


In [3]:
def engineer_features(df, total_charges_median=None):
    df = df.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    median_tc = total_charges_median if total_charges_median is not None else df['TotalCharges'].median()
    df['TotalCharges'] = df['TotalCharges'].fillna(median_tc)
    df['AvgChargesPerMonth'] = df['TotalCharges'] / df['tenure'].replace(0, 1)
    df['MonthlyChargesSqrt'] = np.sqrt(df['MonthlyCharges'])
    df['tenure_squared'] = df['tenure'] ** 2
    df['ChargesRatio'] = df['TotalCharges'] / (df['MonthlyCharges'] * (df['tenure'] + 1))
    df['Contract_MonthToMonth'] = (df['Contract'] == 'Month-to-month').astype(int)
    df['Contract_OneYear'] = (df['Contract'] == 'One year').astype(int)
    df['Contract_TwoYear'] = (df['Contract'] == 'Two year').astype(int)
    df['IsFiberOptic'] = (df['InternetService'] == 'Fiber optic').astype(int)
    df['NumStreaming'] = ((df['StreamingTV'] == 'Yes') | (df['StreamingMovies'] == 'Yes')).astype(int)
    df['NumStreamingBoth'] = ((df['StreamingTV'] == 'Yes') & (df['StreamingMovies'] == 'Yes')).astype(int)
    addon_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
    df['NumAddons'] = sum((df[c] == 'Yes').astype(int) for c in addon_cols)
    df['PaymentElectronic'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
    df['HasDependents'] = (df['Dependents'] == 'Yes').astype(int)
    df['HasPartner'] = (df['Partner'] == 'Yes').astype(int)
    df['SeniorWithShortTenure'] = (df['SeniorCitizen'] == 1) & (df['tenure'] < 12)
    df['SeniorWithShortTenure'] = df['SeniorWithShortTenure'].astype(int)
    df['HighChargeShortTenure'] = (df['MonthlyCharges'] > 70) & (df['tenure'] < 12)
    df['HighChargeShortTenure'] = df['HighChargeShortTenure'].astype(int)
    return df

def target_encode(train_df, test_df, col, target_series, m=5):
    global_mean = target_series.mean()
    combined = pd.DataFrame({col: train_df[col], '_y': target_series.values})
    agg = combined.groupby(col)['_y'].agg(['mean', 'count'])
    smooth = (agg['mean'] * agg['count'] + global_mean * m) / (agg['count'] + m)
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df[f'{col}_te'] = train_df[col].map(smooth).fillna(global_mean)
    test_df[f'{col}_te'] = test_df[col].map(smooth).fillna(global_mean)
    return train_df, test_df

X_full = engineer_features(X_full)
tc_median = X_full['TotalCharges'].median()
X_test_raw = engineer_features(X_test_raw, total_charges_median=tc_median)

if USE_TARGET_ENCODING:
    for col in ['Contract', 'PaymentMethod', 'InternetService']:
        X_full, X_test_raw = target_encode(X_full, X_test_raw, col, y_full, m=10)
        X_full = X_full.drop(columns=[col])
        X_test_raw = X_test_raw.drop(columns=[col])

X_encoded = pd.get_dummies(X_full, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_raw, drop_first=True)
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

In [4]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

In [5]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

xgb_grid = {
    'n_estimators': [300, 500],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'scale_pos_weight': [2],
}
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
gs_xgb = GridSearchCV(xgb, xgb_grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1)
fit_kw = {'sample_weight': sample_weights} if sample_weights is not None else {}
gs_xgb.fit(X_encoded, y_full, **fit_kw)
best_xgb = gs_xgb.best_estimator_
print(f"XGB Best CV AUC: {gs_xgb.best_score_:.4f}")

Fitting 10 folds for each of 8 candidates, totalling 80 fits
XGB Best CV AUC: 0.9165


In [6]:
from lightgbm import LGBMClassifier

lgb_grid = {
    'n_estimators': [300, 500],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'scale_pos_weight': [2],
}
lgb_base = LGBMClassifier(random_state=42, verbose=-1)
gs_lgb = GridSearchCV(lgb_base, lgb_grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1)
gs_lgb.fit(X_encoded, y_full, **fit_kw)
best_lgb = gs_lgb.best_estimator_
print(f"LGB Best CV AUC: {gs_lgb.best_score_:.4f}")

Fitting 10 folds for each of 8 candidates, totalling 80 fits


c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python3

LGB Best CV AUC: 0.9164


In [7]:
from catboost import CatBoostClassifier

cat_grid = {
    'iterations': [300, 500],
    'depth': [4, 6],
    'learning_rate': [0.05, 0.1],
}
cat_base = CatBoostClassifier(random_state=42, verbose=0)
gs_cat = GridSearchCV(cat_base, cat_grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1)
gs_cat.fit(X_encoded, y_full, **fit_kw)
best_cat = gs_cat.best_estimator_
print(f"CatBoost Best CV AUC: {gs_cat.best_score_:.4f}")

Fitting 10 folds for each of 8 candidates, totalling 80 fits
CatBoost Best CV AUC: 0.9165


In [8]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score

n_models = 3 + (2 if USE_RF_ET else 0) + (1 if USE_MLP else 0)
oof = np.zeros((len(X_encoded), n_models))
test_preds = np.zeros((len(X_test_encoded), n_models))
scaler_mlp = StandardScaler()

def get_models(seed, fold):
    models = [
        XGBClassifier(**gs_xgb.best_params_).set_params(random_state=seed+fold, eval_metric='logloss'),
        LGBMClassifier(**gs_lgb.best_params_).set_params(random_state=seed+fold, verbose=-1),
        CatBoostClassifier(iterations=gs_cat.best_params_['iterations'], depth=gs_cat.best_params_['depth'], learning_rate=gs_cat.best_params_['learning_rate'], random_state=seed+fold, verbose=0),
    ]
    i = 3
    if USE_RF_ET:
        models.extend([
            RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
            ExtraTreesClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
        ])
        i += 2
    if USE_MLP:
        models.append(MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, early_stopping=True, random_state=seed+fold))
    return models

for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y_full)):
    X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
    y_tr = y_full.iloc[train_idx]
    sw_tr = sample_weights[train_idx] if sample_weights is not None else None
    models = get_models(42, fold)
    for i, m in enumerate(models):
        if USE_MLP and i == n_models - 1:
            X_tr_s = scaler_mlp.fit_transform(X_tr)
            X_val_s = scaler_mlp.transform(X_val)
            X_test_s = scaler_mlp.transform(X_test_encoded)
            m.fit(X_tr_s, y_tr)
            oof[val_idx, i] = m.predict_proba(X_val_s)[:, 1]
            test_preds[:, i] += m.predict_proba(X_test_s)[:, 1] / N_FOLDS
        else:
            m.fit(X_tr, y_tr, sample_weight=sw_tr) if sw_tr is not None else m.fit(X_tr, y_tr)
            oof[val_idx, i] = m.predict_proba(X_val)[:, 1]
            test_preds[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS

meta = Ridge(alpha=1.0)
meta.fit(oof, y_full)
stack_oof = np.clip(meta.predict(oof), 0, 1)
print(f"Stacking OOF AUC: {roc_auc_score(y_full, stack_oof):.4f}")

Stacking OOF AUC: 0.9165


In [9]:
all_test_preds = [test_preds.copy()]
seeds = [123, 456, 789][:max(0, N_SEEDS-1)]

for base_seed in seeds:
    tp = np.zeros((len(X_test_encoded), n_models))
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y_full)):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr = y_full.iloc[train_idx]
        sw_tr = sample_weights[train_idx] if sample_weights is not None else None
        models = get_models(base_seed, fold)
        for i, m in enumerate(models):
            if USE_MLP and i == n_models - 1:
                scaler_f = StandardScaler()
                X_tr_s = scaler_f.fit_transform(X_tr)
                X_test_s = scaler_f.transform(X_test_encoded)
                m.fit(X_tr_s, y_tr)
                tp[:, i] += m.predict_proba(X_test_s)[:, 1] / N_FOLDS
            else:
                m.fit(X_tr, y_tr, sample_weight=sw_tr) if sw_tr is not None else m.fit(X_tr, y_tr)
                tp[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS
    all_test_preds.append(tp)

final_preds = np.mean([np.clip(meta.predict(tp), 0, 1) for tp in all_test_preds], axis=0)

if USE_CALIBRATION:
    from sklearn.linear_model import LogisticRegression
    calib = LogisticRegression(max_iter=1000, random_state=42)
    calib.fit(stack_oof.reshape(-1, 1), y_full)
    final_preds = calib.predict_proba(final_preds.reshape(-1, 1))[:, 1]

if USE_PSEUDOLABELING:
    X_comb = pd.concat([X_encoded.reset_index(drop=True), X_test_encoded.reset_index(drop=True)])
    y_comb = np.concatenate([y_full.values, (final_preds > 0.5).astype(int)])
    sw_comb = np.concatenate([np.ones(len(y_full)), np.full(len(final_preds), PSEUDO_WEIGHT)])
    oof2 = np.zeros((len(X_comb), n_models))
    tp2 = np.zeros((len(X_test_encoded), n_models))
    meta2 = Ridge(alpha=1.0)
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_comb, y_comb)):
        X_tr, X_val = X_comb.iloc[train_idx], X_comb.iloc[val_idx]
        y_tr, y_val = y_comb[train_idx], y_comb[val_idx]
        sw_tr = sw_comb[train_idx]
        models = get_models(42, fold)
        for i, m in enumerate(models):
            if USE_MLP and i == n_models - 1:
                scaler_f = StandardScaler()
                X_tr_s = scaler_f.fit_transform(X_tr)
                X_test_s = scaler_f.transform(X_test_encoded)
                m.fit(X_tr_s, y_tr)
                oof2[val_idx, i] = m.predict_proba(scaler_f.transform(X_val))[:, 1]
                tp2[:, i] += m.predict_proba(X_test_s)[:, 1] / N_FOLDS
            else:
                m.fit(X_tr, y_tr, sample_weight=sw_tr)
                oof2[val_idx, i] = m.predict_proba(X_val)[:, 1]
                tp2[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS
    meta2.fit(oof2, y_comb)
    final_preds = np.clip(meta2.predict(tp2), 0, 1)

submission = pd.DataFrame({'id': test_ids, 'Churn': final_preds})
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Churn
0,594194,0.031205
1,594195,0.000000
2,594196,0.068309
3,594197,0.000000
4,594198,0.437482
